In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"  # Ajustar se necessário

# ========== FILTRO DE RESTAURANTES E ESTABELECIMENTOS DE COMIDA ==========

# Tags de alimentação
FOOD_TAGS = {
    "Restaurants", "Food", "Fast Food", "Food Trucks", "Food Delivery Services",
    # Culinárias
    "Pizza", "Mexican", "Chinese", "Italian", "Japanese", "Thai", "Vietnamese",
    "Indian", "Greek", "Mediterranean", "French", "Korean", "Filipino", "African",
    "Cuban", "Caribbean", "Middle Eastern", "Latin American", "Asian Fusion",
    "American (Traditional)", "American (New)", "Canadian (New)", "Cajun/Creole",
    "Pakistani", "Southern", "Soul Food", "Tex-Mex",
    # Tipos de estabelecimento
    "Burgers", "Sandwiches", "Sushi Bars", "Seafood", "Steakhouses", "Barbeque",
    "Chicken Wings", "Chicken Shop", "Hot Dogs", "Cheesesteaks", "Tacos",
    "Delis", "Diners", "Buffets", "Breakfast & Brunch",
    "Cafes", "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
    "Juice Bars & Smoothies", "Desserts", "Bakeries", "Donuts", "Bagels",
    "Ice Cream & Frozen Yogurt", "Candy Stores",
    # Bares & bebidas
    "Bars", "Nightlife", "Breweries", "Wine & Spirits", "Beer", "Cocktail Bars",
    "Dive Bars", "Sports Bars", "Pubs", "Gastropubs", "Lounges",
    # Varejo alimentar
    "Grocery", "Specialty Food", "Seafood Markets", "Meat Shops", "Fruits & Veggies",
    "Farmers Market", "Convenience Stores", "Wholesale Stores",
    # Outros
    "Caterers",
}

def is_food_related(categories_str):
    """Verifica se o estabelecimento é relacionado a alimentação."""
    if not categories_str:
        return False
    tags = {t.strip() for t in str(categories_str).split(",")}
    return bool(tags & FOOD_TAGS)

def get_segmento_alimentacao(categories_str):
    """Retorna o segmento de alimentação baseado nas categorias."""
    if not categories_str:
        return "Outros alimentação"
    
    tags = {t.strip() for t in str(categories_str).split(",")}
    
    if "Restaurants" in tags:
        return "RESTAURANTE"
    if tags & {"Bars", "Nightlife", "Breweries", "Cocktail Bars", "Dive Bars",
                "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer"}:
        return "BAR E BEBIDA"
    if tags & {"Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
                "Cafes", "Juice Bars & Smoothies"}:
        return "CAFE"
    if tags & {"Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
                "Candy Stores", "Desserts"}:
        return "PADARIA"
    # Se não se encaixa em nenhum segmento de alimentação, retorna "não são restaurantes"
    segmentos_alimentacao = {
        "Restaurants",
        "Bars", "Nightlife", "Breweries", "Cocktail Bars", "Dive Bars",
        "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer",
        "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
        "Cafes", "Juice Bars & Smoothies",
        "Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
        "Candy Stores", "Desserts"
    }
    if not tags & segmentos_alimentacao:
        return "OUTROS"

# Registra UDFs para uso no Spark
is_food_udf = udf(is_food_related, BooleanType())
get_segmento_udf = udf(get_segmento_alimentacao, StringType())

# ========== FUNÇÕES DE QUALIDADE DE DADOS ==========

def calcular_completude(df: DataFrame, colunas_obrigatorias: list) -> dict:
    """
    Calcula a taxa de completude (% de valores não nulos) para colunas obrigatórias.
    
    Dimensão de Qualidade: COMPLETUDE
    """
    total_registros = df.count()
    metricas = {}
    
    for coluna in colunas_obrigatorias:
        nao_nulos = df.filter(col(coluna).isNotNull()).count()
        taxa_completude = (nao_nulos / total_registros * 100) if total_registros > 0 else 0
        metricas[coluna] = {
            'total': total_registros,
            'preenchidos': nao_nulos,
            'nulos': total_registros - nao_nulos,
            'taxa_completude_%': round(taxa_completude, 2)
        }
    
    return metricas

def validar_precisao_numerica(df: DataFrame, coluna: str, min_val: float = None, max_val: float = None) -> DataFrame:
    """
    Valida se valores numéricos estão dentro de um range esperado.
    
    Dimensão de Qualidade: PRECISÃO
    """
    condicao = col(coluna).isNotNull()
    
    if min_val is not None:
        condicao = condicao & (col(coluna) >= min_val)
    if max_val is not None:
        condicao = condicao & (col(coluna) <= max_val)
    
    return df.filter(condicao)

def remover_duplicados(df: DataFrame, chave_primaria: list, criterio_desempate: str = "hora_ingestao") -> DataFrame:
    """
    Remove duplicados mantendo o registro mais recente baseado no critério de desempate.
    """
    window_spec = Window.partitionBy(chave_primaria).orderBy(col(criterio_desempate).desc())
    df_deduplicated = df.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")
    
    return df_deduplicated

def adicionar_metadados_silver(df: DataFrame) -> DataFrame:
    """
    Adiciona metadados de processamento para camada silver.
    """
    return df.withColumn("data_processamento_silver", 
                         from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))

print("✓ Funções de qualidade de dados carregadas com sucesso!")
print("✓ Filtros de restaurantes e alimentação configurados!")

In [0]:
# ========== TABELA 1: BUSINESS ==========

table_name = "bronze_yelp_academic_dataset_business"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_business = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_business.count()}")

# ========== FILTRO: ESTABELECIMENTOS DE ALIMENTAÇÃO ==========
print("\n--- Filtro de Estabelecimentos de Alimentação ---")
df_business = df_business.filter(is_food_udf(col('categories')))
print(f"Registros relacionados a alimentação: {df_business.count()}")

# Adiciona coluna de food_category
df_business = df_business.withColumn('food_category', get_segmento_udf(col('categories')))
print("✓ Coluna 'food_category' adicionada")

# Mostra distribuição por food_category
print("\nDistribuição por food_category:")
display(df_business.groupBy('food_category').count().orderBy(col('count').desc()))

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'name', 'city', 'state']
metricas_completude = calcular_completude(df_business, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_business_clean = df_business.filter(
    col('business_id').isNotNull() & 
    col('name').isNotNull() & 
    col('city').isNotNull() & 
    col('state').isNotNull()
)

print(f"\nApós filtro de completude: {df_business_clean.count()} registros")

# PRECISÃO: Validação de ranges numéricos
print("\n--- Validação de PRECISÃO ---")

# Stars: 0 a 5
df_business_clean = validar_precisao_numerica(df_business_clean, 'stars', min_val=0, max_val=5)
print(f"  Stars (0-5): {df_business_clean.count()} registros válidos")

# Latitude: -90 a 90
df_business_clean = validar_precisao_numerica(df_business_clean, 'latitude', min_val=-90, max_val=90)
print(f"  Latitude (-90/90): {df_business_clean.count()} registros válidos")

# Longitude: -180 a 180
df_business_clean = validar_precisao_numerica(df_business_clean, 'longitude', min_val=-180, max_val=180)
print(f"  Longitude (-180/180): {df_business_clean.count()} registros válidos")

# Review count: >= 0
df_business_clean = df_business_clean.filter(col('review_count') >= 0)
print(f"  Review count (>=0): {df_business_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_business_clean = remover_duplicados(df_business_clean, ['business_id'])
print(f"Após deduplication: {df_business_clean.count()} registros")

# ========== PADRONIZAÇÃO DO ENDEREÇO ==========
print("\n--- Padronização de Endereços ---")
from pyspark.sql.functions import upper, regexp_replace, trim

# Converte endereço para caixa alta
df_business_clean = df_business_clean.withColumn("address", upper(col("address")))
print("✓ Coluna 'address' convertida para caixa alta")

# ========== PADRONIZAÇÃO DO NOME ==========
print("\n--- Padronização de Nomes ---")

# Apply transformations step by step
name_col = upper(col("name"))
name_col = regexp_replace(name_col, r",.*", "")  # Remove text after comma
name_col = regexp_replace(name_col, r",", " ")  # Replace comma with space
name_col = regexp_replace(name_col, r"\.", " ")  # Replace dot with space
name_col = regexp_replace(name_col, r"ST\.", "SAINT")
name_col = regexp_replace(name_col, r"\bST\b", "SAINT")
name_col = regexp_replace(name_col, r"\bST\.\b", "SAINT")
name_col = regexp_replace(name_col, r"SAINTT", "SAINT")
name_col = regexp_replace(name_col, r"NW ", "NEW")
name_col = regexp_replace(name_col, r"'", " ")  # Replace apostrophes
name_col = regexp_replace(name_col, r"-", " ")  # Replace hyphens
name_col = regexp_replace(name_col, r"^ +| +$", "")  # Trim edges
name_col = regexp_replace(name_col, r" {2,}", " ")  # Collapse multiple spaces
name_col = trim(name_col)

df_business_clean = df_business_clean.withColumn("name_validated", name_col)
print("✓ Coluna 'name_validated' adicionada")

# ========== PADRONIZAÇÃO DA CIDADE ==========
print("\n--- Padronização de Cidades ---")

# Apply transformations step by step
city_col = upper(col("city"))
city_col = regexp_replace(city_col, r",.*", "")  # Remove content after comma
city_col = regexp_replace(city_col, r"/.*", "")  # Remove content after slash
city_col = regexp_replace(city_col, r"BCH", "BEACH")
city_col = regexp_replace(city_col, r",", " ")
city_col = regexp_replace(city_col, r"/", " ")
city_col = regexp_replace(city_col, r"%MTLAUREL%", "MT LAUREL")
city_col = regexp_replace(city_col, r"%TAMPA FLORIDA%", "TAMPA")
city_col = regexp_replace(city_col, r"TWP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"MT \.", "MT")
city_col = regexp_replace(city_col, r"MT\.", "MT")
city_col = regexp_replace(city_col, r"SAINTLOUIS", "SAINT LOUIS")
city_col = regexp_replace(city_col, r"SAINTT", "SAINT")
city_col = regexp_replace(city_col, r"REDINGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"REDNGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"%TWN%", "TOWN")
city_col = regexp_replace(city_col, r"CNTRY", "COUNTRY")
city_col = regexp_replace(city_col, r"NW", "NEW")
city_col = regexp_replace(city_col, r"TOWN & COUNTRY", "TOWN N COUNTRY")
city_col = regexp_replace(city_col, r"\.", " ")  # Replace dot with space
city_col = regexp_replace(city_col, r"ST\.", "SAINT")
city_col = regexp_replace(city_col, r"\bST\b", "SAINT")
city_col = regexp_replace(city_col, r"\bST\.\b", "SAINT")
city_col = regexp_replace(city_col, r"'", " ")  # Replace apostrophes
city_col = regexp_replace(city_col, r"-", " ")  # Replace hyphens
city_col = regexp_replace(city_col, r" {2,}", " ")  # Collapse multiple spaces
city_col = trim(city_col)  # Remove spaces at the beginning and end

df_business_clean = df_business_clean.withColumn("city_validated", city_col)
print("✓ Coluna 'city_validated' adicionada")

# Adiciona metadados silver
df_business_silver = adicionar_metadados_silver(df_business_clean)

# Remove coluna antiga 'segmento' se existir (para evitar duplicação com food_category)
if 'segmento' in df_business_silver.columns:
    df_business_silver = df_business_silver.drop('segmento')
    print("\n✓ Coluna antiga 'segmento' removida")

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_business"
df_business_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_business_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Business:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Métricas gerais
print(f"Total de registros: {df_sample.count()}")
print(f"Campos: {len(df_sample.columns)}")

# Amostra de dados COM name_validated e city_validated
print("\nPrimeiros 10 registros (incluindo padronizações):")
display(df_sample.select(
    'business_id', 
    'name',
    'name_validated',
    'city',
    'city_validated',
    'state', 
    'stars', 
    'review_count',
    'food_category',
    'data_processamento_silver'
).limit(10))

# Estatísticas de qualidade
print("\nEstatísticas de Stars:")
df_sample.select('stars').describe().show()

print("\nDistribuição por Estado (Top 10):")
display(df_sample.groupBy('state').count().orderBy(col('count').desc()).limit(10))

In [0]:
%sql
SELECT 
  name,
  address,
  city_validated,
  state,
  food_category
FROM workspace.yelp_ing.silver_business
WHERE address IS NOT NULL
LIMIT 10

In [0]:
# ========== AMOSTRAS DE PADRONIZAÇÃO DE NOMES ==========

print("ANÁLISE DE PADRONIZAÇÃO DE NOMES")
print("="*80)

df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Amostra geral
print("\n1. Amostra Geral (20 registros):")
print("-"*80)
display(df_business.select('name', 'name_validated', 'city', 'state').limit(20))

# Casos com apóstrofos
print("\n2. Exemplos com Apóstrofos ('):")
print("-"*80)
df_apostrofos = df_business.filter(col('name').contains("'")).select('name', 'name_validated').limit(10)
display(df_apostrofos)

# Casos com hífens
print("\n3. Exemplos com Hífens (-):")
print("-"*80)
df_hifens = df_business.filter(col('name').contains("-")).select('name', 'name_validated').limit(10)
display(df_hifens)

# Casos com "St" ou "St."
print("\n4. Exemplos com 'St' ou 'St.' (transformados para SAINT):")
print("-"*80)
df_saint = df_business.filter(
    col('name').rlike(r"(?i)\bst\.?\b")
).select('name', 'name_validated').limit(10)
display(df_saint)

# Casos mistos (múltiplas transformações)
print("\n5. Casos Complexos (múltiplas transformações):")
print("-"*80)
df_complexos = df_business.filter(
    (col('name').contains("'")) | 
    (col('name').contains("-")) |
    (col('name').rlike(r"(?i)\bst\.?\b"))
).select('name', 'name_validated', 'city').limit(15)
display(df_complexos)

# Estatísticas
print("\n6. Estatísticas de Transformação:")
print("-"*80)
total = df_business.count()
com_apostrofo = df_business.filter(col('name').contains("'")).count()
com_hifen = df_business.filter(col('name').contains("-")).count()
com_st = df_business.filter(col('name').rlike(r"(?i)\bst\.?\b")).count()

print(f"Total de estabelecimentos: {total:,}")
print(f"Nomes com apóstrofo ('): {com_apostrofo:,} ({com_apostrofo/total*100:.1f}%)")
print(f"Nomes com hífen (-): {com_hifen:,} ({com_hifen/total*100:.1f}%)")
print(f"Nomes com 'St' ou 'St.': {com_st:,} ({com_st/total*100:.1f}%)")

print("\n✓ Análise de padronização concluída!")

In [0]:
# ========== TABELA 2: REVIEW ==========

table_name = "bronze_yelp_academic_dataset_review"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_review = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_review.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['review_id', 'user_id', 'business_id', 'stars', 'text', 'date']
metricas_completude = calcular_completude(df_review, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_review_clean = df_review.filter(
    col('review_id').isNotNull() & 
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() &
    col('stars').isNotNull() &
    col('text').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_review_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Stars: 1 a 5 (reviews não permitem 0 stars)
df_review_clean = validar_precisao_numerica(df_review_clean, 'stars', min_val=1, max_val=5)
print(f"  Stars (1-5): {df_review_clean.count()} registros válidos")

# Texto não vazio (após trim)
df_review_clean = df_review_clean.filter(length(trim(col('text'))) > 0)
print(f"  Texto não vazio: {df_review_clean.count()} registros válidos")

# Useful, funny, cool >= 0
for coluna in ['useful', 'funny', 'cool']:
    df_review_clean = df_review_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (useful/funny/cool >=0): {df_review_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_review_clean = remover_duplicados(df_review_clean, ['review_id'])
print(f"Após deduplication: {df_review_clean.count()} registros")

# Adiciona metadados silver
df_review_silver = adicionar_metadados_silver(df_review_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_review"
df_review_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_review_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Review:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")

print(f"Total de reviews: {df_sample.count()}")

# Amostra de dados
print("\nPrimeiros 5 registros:")
display(df_sample.select(
    'review_id', 
    'user_id', 
    'business_id', 
    'stars',
    'date',
    'data_processamento_silver'
).limit(5))

# Distribuição de stars
print("\nDistribuição de Stars:")
display(df_sample.groupBy('stars').count().orderBy('stars'))

In [0]:
# ========== TABELA 3: USER ==========

table_name = "bronze_yelp_academic_dataset_user"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_user = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_user.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'name', 'yelping_since']
metricas_completude = calcular_completude(df_user, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_user_clean = df_user.filter(
    col('user_id').isNotNull() & 
    col('name').isNotNull() & 
    col('yelping_since').isNotNull()
)

print(f"\nApós filtro de completude: {df_user_clean.count()} registros")

# PRECISÃO: Validações numéricas
print("\n--- Validação de PRECISÃO ---")

# Average stars: 0 a 5
df_user_clean = validar_precisao_numerica(df_user_clean, 'average_stars', min_val=0, max_val=5)
print(f"  Average stars (0-5): {df_user_clean.count()} registros válidos")

# Review count, fans, useful, funny, cool >= 0
for coluna in ['review_count', 'fans', 'useful', 'funny', 'cool']:
    df_user_clean = df_user_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (>=0): {df_user_clean.count()} registros válidos")

# Compliments >= 0
compliment_cols = [c for c in df_user_clean.columns if c.startswith('compliment_')]
for coluna in compliment_cols:
    df_user_clean = df_user_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Compliments (>=0): {df_user_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_user_clean = remover_duplicados(df_user_clean, ['user_id'])
print(f"Após deduplication: {df_user_clean.count()} registros")

# Adiciona metadados silver
df_user_silver = adicionar_metadados_silver(df_user_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_user"
df_user_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_user_silver.count()}")
print("="*60)

In [0]:
# ========== TABELA 4: TIP ==========

table_name = "bronze_yelp_academic_dataset_tip"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_tip = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_tip.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'business_id', 'text', 'date']
metricas_completude = calcular_completude(df_tip, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_tip_clean = df_tip.filter(
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() & 
    col('text').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_tip_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Texto não vazio
df_tip_clean = df_tip_clean.filter(length(trim(col('text'))) > 0)
print(f"  Texto não vazio: {df_tip_clean.count()} registros válidos")

# Compliment count >= 0
df_tip_clean = df_tip_clean.filter(
    col('compliment_count').isNull() | (col('compliment_count') >= 0)
)
print(f"  Compliment count (>=0): {df_tip_clean.count()} registros válidos")

# Remoção de duplicados (chave composta: user_id + business_id + date + text)
print("\n--- Remoção de Duplicados ---")
df_tip_clean = remover_duplicados(df_tip_clean, ['user_id', 'business_id', 'date', 'text'])
print(f"Após deduplication: {df_tip_clean.count()} registros")

# Adiciona metadados silver
df_tip_silver = adicionar_metadados_silver(df_tip_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_tip"
df_tip_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_tip_silver.count()}")
print("="*60)

In [0]:
# ========== TABELA 5: CHECKIN ==========

table_name = "bronze_yelp_academic_dataset_checkin"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_checkin = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_checkin.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'date']
metricas_completude = calcular_completude(df_checkin, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_checkin_clean = df_checkin.filter(
    col('business_id').isNotNull() & 
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_checkin_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Date não vazio (campo contém timestamps separados por vírgula)
df_checkin_clean = df_checkin_clean.filter(length(trim(col('date'))) > 0)
print(f"  Date não vazio: {df_checkin_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_checkin_clean = remover_duplicados(df_checkin_clean, ['business_id'])
print(f"Após deduplication: {df_checkin_clean.count()} registros")

# Adiciona metadados silver
df_checkin_silver = adicionar_metadados_silver(df_checkin_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_checkin"
df_checkin_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_checkin_silver.count()}")
print("="*60)

In [0]:
# ========== RESUMO FINAL: QUALIDADE DE DADOS SILVER ==========

print("RELATÓRIO DE QUALIDADE - CAMADA SILVER")
print("="*80)
print("\nDimensões de Qualidade Aplicadas:")
print("  1. COMPLETUDE: Campos obrigatórios preenchidos")
print("  2. PRECISÃO: Valores dentro dos ranges esperados\n")
print("="*80)

# Lista de tabelas silver
tabelas_silver = [
    'silver_business',
    'silver_review',
    'silver_user',
    'silver_tip',
    'silver_checkin'
]

resumo = []

for tabela in tabelas_silver:
    try:
        df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{tabela}")
        count = df.count()
        colunas = len(df.columns)
        
        # Verifica se tem o campo de processamento
        tem_metadata = 'data_processamento_silver' in df.columns
        
        resumo.append({
            'Tabela': tabela,
            'Registros': count,
            'Colunas': colunas,
            'Metadata Silver': '✓' if tem_metadata else '✗'
        })
        
    except Exception as e:
        print(f"Erro ao processar {tabela}: {str(e)}")

# Cria DataFrame de resumo
import pandas as pd
df_resumo = pd.DataFrame(resumo)

print("\nTABELAS SILVER CRIADAS:")
print(df_resumo.to_string(index=False))

print("\n" + "="*80)
print("\nPRÓXIMOS PASSOS RECOMENDADOS:")
print("  1. Validar relacionamentos entre tabelas (FKs)")
print("  2. Criar testes de qualidade automatizados")
print("  3. Implementar monitoramento contínuo")
print("  4. Documentar regras de negócio aplicadas")
print("  5. Criar camada Gold com agregações analíticas")
print("\n" + "="*80)
print("\n✓ Processo de limpeza concluído com sucesso!")